### Imports

In [28]:
import numpy as np
import pandas as pd
import sklearn as skl
import skimage as ski
from tqdm import tqdm, trange
import matplotlib.pyplot as plt

### Constant, Variable, and DataFrame Instatiation

In [29]:
PALLET_DIMS = (1000, 1400, 1400)                                                # Length, width, height (X, Y, Z, respectively) in mm
ML_OBSERVATION_SCALE_FACTOR = 10                                                # Scale factor to scale down height map observations for ML model input

boxtypes = pd.read_csv("boxtypes.csv")                                          # Load box type dimensions
orders = pd.read_csv("orders.csv")                                              # Load orders data

#### Environment Definition

In [30]:
class Pallet:
    def __init__(self, dims=PALLET_DIMS):                                       # Initialize pallet with given dimensions
        # unpack pallet dimensions
        self.size_x, self.size_y, self.size_z = dims
        # Initialize list to store boxes placed on the pallet
        self.boxes = []
        # Initialize heightmap to track box heights at each (x, y) position
        self.heightmap = np.zeros((self.size_x, self.size_y), dtype=np.int32)

    def reset(self):                                                            # Emtpy the pallet
        # Clear the list of boxes
        self.boxes = []
        # Reset the heightmap to all zeros
        self.heightmap = np.zeros((self.size_x, self.size_y))

    def get_max_height(self):                                                   # Max height of boxes on the pallet, value to optimize for
        return np.max(self.heightmap)
    
    def get_max_height_in_area(self, x, y, dx, dy):                             # Get the maximum height in a rectangular area of the heightmap
        # Ensure we don't go out of bounds
        x_end = min(x + dx, self.size_x)                                          
        y_end = min(y + dy, self.size_y)

        # Specify the rectangle being checked
        region = self.heightmap[x:x_end, y:y_end]

        # Return -1 (fail) if the region is of size 0 or smaller
        if region.size <= 0:
            return -1
        
        # Otherwise, return the maximum height in the region
        return np.max(region)
    
    def get_in_bounds_status(self, x, y, z):                                    # Check if the given (x, y, z) position is within the pallet boundaries
        return (x <= self.size_x) and (y <= self.size_y) and (z <= self.size_z)
    
    def place_box(self, box_dims, x, y):                                        # Attempt to place a box of given dimensions at (x, y) position on the pallet
        # Unpack box dimensions
        dx, dy, dz = box_dims

        # Make sure x and y are ints so we don't get partial millimeters from the model
        x = int(x)
        y = int(y)

        # Get the height the bottom of the box will be
        z = self.get_max_height_in_area(x, y, dx, dy)

        # Check if the box fits within the pallet boundaries, or return False
        if not self.get_in_bounds_status(x+dx, y+dy, z+dz):
            return False

        # Place the box by updating the heightmap
        self.heightmap[x:x+dx, y:y+dy] = z + dz

        # Store the box's position and dimensions
        self.boxes.append({
            'x': x, 'y': y, 'z': z, 'dx': dx, 'dy': dy, 'dz': dz
        })

        # Return True to indicate successful placement
        return True
    
    def visualize_heightmap(self, pallet_id):                                   # Visualize the heightmap
        plt.imshow(self.heightmap.T, origin='lower', cmap='viridis', extent=[0, self.size_x, 0, self.size_y])
        plt.colorbar(label='Height (mm)')
        plt.title(f'Heightmap of Pallet: {pallet_id}')
        plt.xlabel('X (mm)')
        plt.ylabel('Y (mm)')
        plt.show()

    def visualize_boxes(self, pallet_id):                                       # Visualize the 3d boxes on the pallet using matplotlib
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')

        for box in self.boxes:
            # Create a 3D block for each box
            ax.bar3d(box['x'], box['y'], box['z'], box['dx'], box['dy'], box['dz'], alpha=0.7)

        ax.set_xlabel('X (mm)')
        ax.set_ylabel('Y (mm)')
        ax.set_zlabel('Z (mm)')
        ax.set_title(f'Boxes on Pallet: {pallet_id}')
        plt.show()

    def get_observation(self, scale_factor=ML_OBSERVATION_SCALE_FACTOR):        # Get a downscaled resolution view of the height map for ML
        # Use block_reduce from scikit-image to scale down heightmap, making sure to take the max so the model knows the top.
        observation = ski.measure.block_reduce(self.heightmap, block_size=(scale_factor, scale_factor), func=np.max)
        return observation

#### Function Definitions

In [ ]:
def get_box_properties_from_id(boxid, df=boxtypes):                             # Retrieve length (x), width (y), height (z), area (a) and volume (v) of a box from its ID
    box = df[df['ID'] == boxid]

    x = box.iloc[0]['LENGTH']
    y = box.iloc[0]['WIDTH']
    z = box.iloc[0]['HEIGHT']

    a = x * y
    v = a * z
    
    return x, y, z, a, v

def get_box_list_from_order(orderid, df=orders):                                # Retrieve list of box IDs from a given order ID
    order = df[df['order_id'] == orderid]
    
    box_list = []

    for i in range(1, 11):
        column = f'amt_{i}'
        if column in order.columns:
            amount = order.iloc[0][column]
            for j in range(amount):
                box_list.append(i)

    return box_list

def sort_order_by_size(box_list, criterion="v", invert=False):                  # Return a list of box IDs sorted by size (default = largest to smallest). Arguments: invert (bool) to sort smallest to largest, sortby (str) to choose sorting criterion (length ("x"), width ("y"), height ("z"), area ("a"), volume ("v"))
    # The get_box_properties_from_id function returns (x, y, z, a, v), so map sortby to the correct index of those outputs
    criterion_to_index_dict = {
        "x": 0, "length": 0,
        "y": 1, "width": 1,
        "z": 2, "height": 2,
        "a": 3, "area": 3,
        "v": 4, "volume": 4
    }

    # Make sure no bogus value is used
    if criterion not in criterion_to_index_dict:
        raise ValueError("Invalid sortby value. Use 'x', 'y', 'z', 'a', or 'v'.")

    # Sort the box list based on the chosen criterion with lambda function
    sorted_list = sorted(
        box_list, 
        key=lambda boxid: get_box_properties_from_id(boxid)[criterion_to_index_dict[criterion]], 
        reverse=not invert
    )
    
    return sorted_list

In [35]:
box_list = get_box_list_from_order(1)
print(box_list)
print(sort_order_by_size(box_list))

[1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 10, 10, 10]
[3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 10, 10, 10, 6, 6, 6, 6, 1, 1, 1, 1]


#### Box Placing Algorithms

In [ ]:
def place_random_boxes(pallet, num_boxes, box_size_range):         # Place a number of randomly sized boxes on the pallet
    for _ in range(num_boxes):
        # Generate random box dimensions within the specified range
        dx = np.random.randint(box_size_range[0], box_size_range[1])
        dy = np.random.randint(box_size_range[0], box_size_range[1])
        dz = np.random.randint(box_size_range[0], box_size_range[1])

        # Generate random (x, y) position for the box
        x = np.random.randint(0, pallet.size_x)
        y = np.random.randint(0, pallet.size_y)

        # Attempt to place the box on the pallet
        pallet.place_box((dx, dy, dz), x, y)

#### Testing Area

In [39]:
# first = Pallet()
# place_boxes_random(first, num_boxes=100, box_size_range=(100, 400))
# first.visualize_heightmap("Random")
# first.visualize_boxes("Random")

boxid = 1
x, y, z, a, v = get_box_properties_from_id(boxid)
print(f"Sizes of box {boxid}: X = {x}, Y = {y}, Z = {z}, A = {a}, V = {v}")

orderid = 1
box_list = get_box_list_from_order(orderid)
criterion = "volume"
print(f"Box list for order {orderid}: {box_list}")
print(f"Same list sorted by box {criterion}: {sort_order_by_size(box_list)}")

Sizes of box 1: X = 349, Y = 246, Z = 62, A = 85854, V = 5322948
Box list for order 1: [1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 10, 10, 10]
Same list sorted by box volume: [3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 10, 10, 10, 6, 6, 6, 6, 1, 1, 1, 1]
